# 06 · Computational cost

This notebook measures how much computing time the final setting needs compared with the baseline, for each of the three foundation models, on the graphics card of the computer that runs it.
The paper reports these times in the Results section *Computational cost of the final setting compared with the baseline* and in Supplementary Fig. 3, which notebook 08 draws from the files written here.

| | Baseline | Final setting |
|---|---|---|
| Adaptation (done once, before the model is used) | one fine-tuning run | a teacher run, the calculation of the teacher outputs, a student run, and the fitting of the penalized head |
| Reading a segment | one forward pass of the segment, model head | mixed reading: one forward pass of the segment and one of each of its 25 windows of 2.5 s (26 passes), penalized head |

Three times are measured for each foundation model and each setting:

1. the **time of adaptation**, each part measured once;
2. the **time to read one segment**, when segments are read one at a time;
3. the **time to read 100,000 segments** with a batch size of 512, calculated from the time for each segment that is measured when segments are read in batches.

**Output files** (in `TIMING`, described in the last section): `adaptation_time.csv` and `reading_time.csv`.

**Time needed.** The adaptation part trains nine models (a baseline, a teacher and a student for each foundation model) with the full 3,000 steps, because the duration of these runs is what it measures.
In the paper, on one NVIDIA RTX 3080 (10 GB), adaptation took 5–24 min for the baseline and 15–65 min for the final setting, depending on the foundation model:

| | Baseline | Final setting |
|---|---|---|
| ECGFounder | 5 min | 15 min |
| HuBERT-ECG | 15 min | 52 min |
| ECG-FM | 24 min | 65 min |

The complete measurement therefore takes about **3 h of GPU time**. The reading part takes about 4 min of it.
With `QUICK_TEST = True` the adaptation part runs only 40 steps on a few hundred recordings (about 3 min), to show that the code works; its times are written to a file of their own and are not measurements.
The reading times are measured in full in both modes, because they do not depend on how long the model was trained.

**Conditions.** A time is meaningful only if nothing else uses the graphics card, and nothing else loads the processor or the disk heavily, while it is measured.
The times in the paper were measured with mixed precision and one job at a time, with no other work running on the computer. Every time of adaptation is one run with seed 0, so its variation between runs is not known.

**Processor cores.** The time to read one segment at a time is set mainly by the processor, not by the graphics card (see *One segment at a time* below).
Some processors have fast and slow cores, and the operating system may run a process that works in the background on the slow cores.
This can make the time to read one segment more than twice as long, while the time to read many segments in batches does not change.
If the times for one segment are longer than expected, set `CPU_CORES` below to the numbers of the fast logical processors
(for example `list(range(16))` on a processor whose first 16 logical processors are the two threads of each of 8 fast cores). With `CPU_CORES = None` the choice is left to the operating system.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
QUICK_TEST = False        # True: functional check. The adaptation part runs 40 steps on a few hundred recordings and writes its times to a file of its own (they are NOT measurements); the reading times are measured in full
CPU_CORES = None          # None: the operating system chooses the processor cores. A list of logical processors, for example list(range(16)), restricts this notebook to them (see "Processor cores" above)

if CPU_CORES is not None:
    import psutil                                 # installed with Jupyter (a requirement of ipykernel)
    psutil.Process().cpu_affinity(list(CPU_CORES))
    print("this process is restricted to the logical processors", list(CPU_CORES))

TIMING_SEED = 0                                   # seed of every run of the measurement
STEPS, EVERY, SUBSET = (40, 20, QUICK_SUBSET) if QUICK_TEST else (TRAIN_STEPS, EVAL_EVERY, None)
SUFFIX = "_quick_test" if QUICK_TEST else ""
RUN_DIR = TIMING / f"adaptation_runs{SUFFIX}"     # everything that the timed runs write: models, training logs, teacher outputs, penalized heads
MODEL_DIR, LOG_DIR, TARGET_DIR, HEAD_DIR = RUN_DIR / "models", RUN_DIR / "training_logs", RUN_DIR / "teacher_targets", RUN_DIR / "penalized_heads"
ADAPTATION_FILE = TIMING / f"adaptation_time{SUFFIX}.csv"
READING_FILE = TIMING / "reading_time.csv"
for _d in (MODEL_DIR, LOG_DIR, TARGET_DIR, HEAD_DIR):
    _d.mkdir(parents=True, exist_ok=True)
assert DEVICE.type == "cuda", "this notebook needs a GPU"
GPU_NAME = f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2 ** 30:.0f} GB)"
print("GPU:", GPU_NAME, "| mixed precision:", AMP, "| the timed runs write to", RUN_DIR)

## Time of adaptation

### What is timed

The functions that are timed are the ones that notebooks 03 and 04 use to adapt the models (`finetune`, `compute_teacher_targets`, `fit_penalized_heads`, all defined in notebook 00),
with the settings of the study, so the times are those of the method as it is run.

| Setting | Part | What is run | The time includes |
|---|---|---|---|
| baseline | `fine_tuning` | `finetune(model, "baseline", seed)`: 3,000 steps of 128 labelled 5-s segments | building the model from the pretrained weights, reading the training segments from disk, the validation every 250 steps, saving the selected weights |
| final | `teacher_run` | `finetune(model, "teacher", seed)`: 3,000 steps of 128 labelled 2.5-s windows | as above |
| final | `teacher_outputs` | `compute_teacher_targets`: the teacher reads seven 2.5-s windows of each of the 34,168 training segments and the 42,722 segments of the unlabelled pool | loading the teacher, reading the recordings from disk, saving the outputs |
| final | `student_run` | `finetune(model, "student", seed)`: 3,000 steps of 128 labelled and 128 unlabelled 5-s segments | as for the baseline |
| final | `head_fitting` | `fit_penalized_heads(student, gammas=[PENALTY_WEIGHT])`: embeddings of every training segment and of its two 4.5-s views (graphics card), then five logistic regressions (processor) | reading the recordings from disk; not the loading of the student from disk |

The parts of the final setting follow each other, because each needs the output of the part before it. The time of a setting is the sum of its parts.
Notebook 04 fits the head for five penalty weights at once, to study the effect of the weight. The final setting needs only the weight `PENALTY_WEIGHT` (300), so one head is fitted here.

The validation during a run is part of adaptation as it is done in the study (it selects the weights that are saved), so it is counted. With another number of steps the time of a fine-tuning run changes almost in proportion;
the times of the teacher outputs and of the head do not depend on the number of steps.

### Where the timed runs write

The runs write their models, training logs, teacher outputs and heads to the folder `adaptation_runs` in `TIMING`, never to `FINETUNED` or `TEACHER_TARGETS`: the models of the study are not replaced by the models of a timing run.
The folder takes about 2.4 GB (nine models). The reading part below uses its models; after the notebook has finished the folder can be deleted, because notebook 08 needs only the two CSV files.

### Resuming

The time of each part is written to `adaptation_time.csv` as soon as the part has finished. A part whose time is in the file and whose output exists is skipped, so an interrupted measurement can be continued.
When a part is run, the parts of the same setting that follow it are run as well, because they use its output. To repeat the whole measurement, delete `adaptation_time.csv`.

In [ ]:
# the parts of the adaptation in each setting, in the order in which they are run
PARTS = {"baseline": ["fine_tuning"], "final": ["teacher_run", "teacher_outputs", "student_run", "head_fitting"]}
RUN_KIND = {"fine_tuning": "baseline", "teacher_run": "teacher", "student_run": "student"}      # the parts that are fine-tuning runs, and the kind of model that each trains


def part_output(model, part):
    """The file that a part of the adaptation writes."""
    baseline, teacher, student = (model_tag(model, kind, TIMING_SEED) for kind in ("baseline", "teacher", "student"))
    return {"fine_tuning": MODEL_DIR / f"{baseline}.pt", "teacher_run": MODEL_DIR / f"{teacher}.pt", "teacher_outputs": TARGET_DIR / f"{teacher}.npz",
            "student_run": MODEL_DIR / f"{student}.pt", "head_fitting": HEAD_DIR / f"{student}.npz"}[part]


def run_part(model, part):
    """Run one part of the adaptation of one foundation model once. Returns its time in minutes and the peak GPU memory in GB."""
    teacher, student = model_tag(model, "teacher", TIMING_SEED), model_tag(model, "student", TIMING_SEED)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    if part in RUN_KIND:
        info = finetune(model, RUN_KIND[part], TIMING_SEED, steps=STEPS, eval_every=EVERY, subset=SUBSET, model_dir=MODEL_DIR, log_dir=LOG_DIR, target_dir=TARGET_DIR)
        return info["minutes"], info["peak_gpu_gb"]
    if part == "teacher_outputs":
        minutes = compute_teacher_targets(teacher, subset=SUBSET, model_dir=MODEL_DIR, target_dir=TARGET_DIR)["minutes"]
    else:                                         # head_fitting
        net = load_finetuned(student, MODEL_DIR)
        heads = fit_penalized_heads(net, gammas=[PENALTY_WEIGHT], n_train=SUBSET[0] if SUBSET else None, verbose=False)
        save_heads(heads, student, HEAD_DIR)      # kept for the reading part below
        minutes = (heads["seconds_embeddings"] + heads["seconds_fit"]) / 60
        del net
    return minutes, torch.cuda.max_memory_allocated() / 2 ** 30


def load_measured():
    """The times of the parts that were measured before: {(model, setting, part): {"minutes", "peak_gpu_gb", "gpu"}}."""
    if not ADAPTATION_FILE.exists():
        return {}
    T = pd.read_csv(ADAPTATION_FILE); T = T[T.part != "total"]
    return {(r.model, r.setting, r.part): {"minutes": r.minutes, "peak_gpu_gb": r.peak_gpu_gb, "gpu": r.gpu} for r in T.itertuples()}


def adaptation_table(measured):
    """The table that is written to ADAPTATION_FILE: one row for each measured part, and a row "total" for each setting whose parts are all measured."""
    rows = []
    for model in MODELS:
        totals = {}
        for setting, parts in PARTS.items():
            have = [p for p in parts if (model, setting, p) in measured]
            rows += [{"model": model, "setting": setting, "part": p, **measured[(model, setting, p)]} for p in have]
            if have == parts:
                totals[setting] = sum(measured[(model, setting, p)]["minutes"] for p in parts)
                rows.append({"model": model, "setting": setting, "part": "total", "minutes": totals[setting],
                             "ratio_to_baseline": totals[setting] / totals["baseline"] if "baseline" in totals else np.nan,
                             "peak_gpu_gb": max(measured[(model, setting, p)]["peak_gpu_gb"] for p in parts), "gpu": measured[(model, setting, parts[-1])]["gpu"]})
    T = pd.DataFrame(rows, columns=["model", "setting", "part", "minutes", "ratio_to_baseline", "peak_gpu_gb", "gpu"])
    T["quick_test"] = QUICK_TEST
    return T

### Measurement

Each part is run once. On one RTX 3080 the cell takes about 20 min for ECGFounder, 68 min for HuBERT-ECG and 88 min for ECG-FM.

In [ ]:
MEASURED = load_measured()
for model in MODELS:                              # about 3 h on one RTX 3080 (with QUICK_TEST about 3 min)
    for setting, parts in PARTS.items():
        again = False                             # set when a part is run: the parts of the setting that follow it are then run as well
        for part in parts:
            key = (model, setting, part)
            if not again and key in MEASURED and part_output(model, part).exists():
                print(f"[{model}] {setting} / {part}: {MEASURED[key]['minutes']:.1f} min, measured before: skipped", flush=True); continue
            again = True
            minutes, peak_gb = run_part(model, part)
            MEASURED[key] = {"minutes": minutes, "peak_gpu_gb": peak_gb, "gpu": GPU_NAME}
            adaptation_table(MEASURED).to_csv(ADAPTATION_FILE, index=False, float_format="%.6g")
            print(f"[{model}] {setting} / {part}: {minutes:.1f} min | peak GPU memory {peak_gb:.2f} GB", flush=True)

ADAPTATION = adaptation_table(MEASURED)
ADAPTATION.to_csv(ADAPTATION_FILE, index=False, float_format="%.6g")
ADAPTATION.round(2)

## Time of reading

### What is timed

- **Baseline:** `read_probabilities(net, x)`: one forward pass of each segment, model head.
- **Final setting:** `read_probabilities(net, x, heads, gamma=PENALTY_WEIGHT, mixed=True)`: one forward pass of each segment, the 25 windows of 2.5 s cut from it and passed through the network,
  the penalized head, and the mixing of the two probabilities (26 forward passes for a segment).

**Inputs.** The inputs are random numbers of the size of a segment (12 leads, 2,500 samples) that are already on the graphics card. The time of a forward pass depends on the size of the input and not on its values.
Reading the recordings from disk and filtering them is not counted; it is the same for both settings. The scaling of the input that each model applies (notebook 00) is part of the model and is counted.

**Network.** Both ways of reading are timed with the same network, the student of the adaptation part above, with its model head for the baseline reading and its penalized head for the final setting.
The baseline model has the same architecture as the student, so a forward pass takes the same time in both, and the time of a read does not depend on the values of the weights.
For this reason the reading times are real measurements also with `QUICK_TEST = True`, where the student was trained for a few steps only.

**Clock.** The graphics card works asynchronously: a call returns before the result is ready. The clock is therefore read only after `torch.cuda.synchronize()`, when all the work that was sent to the card has finished.
Mixed precision is used as everywhere else.

**Memory.** `peak_gpu_gb` is the largest amount of memory that PyTorch had allocated on the graphics card during a reading (weights, inputs and activations).
Memory that was allocated before the model was loaded is not counted, and the memory that the CUDA libraries themselves need is not part of it.

In [ ]:
READ_BATCH = 512                                  # batch size when many segments are read
N_SEGMENTS = 100_000                              # the time of reading many segments is given for this number of segments (it is part of the name of the column minutes_per_100000_segments)
ROUNDS, RUNS, WARMUP = 7, 100, 10                 # one segment at a time: rounds; timed reads in a round; reads before them that are not timed
TIMED_SEGMENTS, REPEATS = 1024, 3                 # many segments: random segments that are read in one repeat; repeats


def seconds(fn):
    """Time of one call of fn, with the clock read only when the graphics card has finished its work."""
    torch.cuda.synchronize(); t0 = time.perf_counter(); fn(); torch.cuda.synchronize()
    return time.perf_counter() - t0


def random_segments(n):
    """n random inputs of the size of a 5-s segment, on the graphics card."""
    return torch.randn(n, 12, SEG, device=DEVICE, generator=torch.Generator(device=DEVICE).manual_seed(0))


def two_readings(net, heads, x):
    """The baseline reading and the reading of the final setting of the segments x, as functions without arguments."""
    return {"baseline": lambda: read_probabilities(net, x, batch_size=READ_BATCH),
            "final": lambda: read_probabilities(net, x, heads=heads, gamma=PENALTY_WEIGHT, mixed=True, batch_size=READ_BATCH)}

### One segment at a time

One segment is read, as when ECGs arrive one by one. In the final setting the segment is passed through the network first, and its 25 windows are then passed together as one batch.

The measurement has 7 rounds. In a round, each way of reading is called 10 times without timing (warm-up) and then 100 times with timing, and the median of the 100 times is the time of the round.
The two ways of reading alternate within every round, so that both are measured under the same state of the computer.
The time that is reported is the median over the rounds, and the ratio to the baseline is the median over the rounds of the ratio within a round.

With one segment the graphics card is far from full, so the batch of 25 windows takes little longer than a single input: the final setting takes a few times as long as the baseline, not 26 times.
Most of the time of such a small forward pass is spent by the processor, which sends the operations of the network to the graphics card one after the other.
The time for one segment therefore depends on the speed of the processor core on which the notebook runs (see *Processor cores* at the top), and the time for many segments does not.

In [ ]:
def time_one_segment(net, heads):
    """Milliseconds to read one segment with the baseline and with the final setting, the ratio, and the peak GPU memory: a list of two rows."""
    readings = two_readings(net, heads, random_segments(1))
    ms = {setting: [] for setting in readings}
    for _ in range(ROUNDS):
        for setting, read in readings.items():
            for _ in range(WARMUP):
                read()
            ms[setting].append(1000 * float(np.median([seconds(read) for _ in range(RUNS)])))
    ratio = {"baseline": 1.0, "final": float(np.median(np.array(ms["final"]) / np.array(ms["baseline"])))}
    rows = []
    for setting, read in readings.items():
        torch.cuda.reset_peak_memory_stats(); read(); torch.cuda.synchronize()
        rows.append({"task": "one_segment", "setting": setting, "segments_per_batch": 1, "milliseconds_per_segment": float(np.median(ms[setting])),
                     "ratio_to_baseline": ratio[setting], "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30})
    return rows

### Many segments in batches

When many stored ECGs are read, the inputs are passed through the network in batches of 512. In the final setting this holds for the windows as well: the 25 windows of 512 segments are 12,800 inputs, which are passed in 25 batches of 512.

1,024 random segments are read three times with each way of reading, after one batch of segments and one batch of windows has been passed without timing (warm-up).
The time for each segment is the median of the three times divided by 1,024, and the time for 100,000 segments is the time for each segment multiplied by 100,000.
The ratio to the baseline is the ratio of the two times for each segment.

In batches the graphics card is fully used, so the time grows with the amount of signal that is read. A 2.5-s window costs about half of a 5-s segment in ECGFounder and ECG-FM,
which gives a ratio of about 1 + 25 × ½ ≈ 14. HuBERT-ECG receives a window repeated to the length of 5 s (notebook 00), so a window costs as much as a segment and the ratio is about 26.

In [ ]:
def time_many_segments(net, heads):
    """Milliseconds for each segment when segments are read in batches of READ_BATCH, with the baseline and with the final setting, the ratio, and the peak GPU memory: a list of two rows."""
    x = random_segments(TIMED_SEGMENTS)
    read_probabilities(net, x[:READ_BATCH], batch_size=READ_BATCH); read_probabilities(net, x[:READ_BATCH, :, :WIN], batch_size=READ_BATCH)      # warm-up: one batch of segments, one batch of windows
    rows = []
    for setting, read in two_readings(net, heads, x).items():
        torch.cuda.reset_peak_memory_stats()
        ms = 1000 * float(np.median([seconds(read) for _ in range(REPEATS)])) / TIMED_SEGMENTS
        rows.append({"task": "many_segments", "setting": setting, "segments_per_batch": READ_BATCH, "milliseconds_per_segment": ms, "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30})
    for r in rows:
        r["ratio_to_baseline"] = r["milliseconds_per_segment"] / rows[0]["milliseconds_per_segment"]
        r["minutes_per_100000_segments"] = r["milliseconds_per_segment"] * N_SEGMENTS / 60000
    return rows

### Measurement

About 4 min for the three models on one RTX 3080. The measurement is repeated whenever the cell is run, and `reading_time.csv` is replaced.

In [ ]:
_rows = []
for model in MODELS:
    assert part_output(model, "student_run").exists() and part_output(model, "head_fitting").exists(), f"the adaptation part above has not finished for {model}"
    tag = model_tag(model, "student", TIMING_SEED)
    torch.cuda.empty_cache(); _before = torch.cuda.memory_allocated() / 2 ** 30       # memory that earlier cells left allocated is not counted as memory of the reading
    net, heads = load_finetuned(tag, MODEL_DIR), load_heads(tag, HEAD_DIR)
    for r in time_one_segment(net, heads) + time_many_segments(net, heads):
        r["peak_gpu_gb"] -= _before
        _rows.append({"model": model, **r, "forward_passes": 1 + N_WIN_READ if r["setting"] == "final" else 1, "gpu": GPU_NAME})
        print(f"[{model}] {r['task']:13s} {r['setting']:8s} {r['milliseconds_per_segment']:8.3f} ms for a segment | ratio to baseline {r['ratio_to_baseline']:5.2f} | peak GPU memory {r['peak_gpu_gb']:.2f} GB", flush=True)
    del net; torch.cuda.empty_cache()

READING = pd.DataFrame(_rows, columns=["model", "task", "setting", "forward_passes", "segments_per_batch", "milliseconds_per_segment", "ratio_to_baseline", "minutes_per_100000_segments", "peak_gpu_gb", "gpu"])
READING.to_csv(READING_FILE, index=False, float_format="%.6g")
READING.round(3)

## Summary

The table collects the numbers that Supplementary Fig. 3 shows: for each foundation model the time of the baseline, the time of the final setting and their ratio, for adaptation, for one segment and for 100,000 segments.

In the paper (one NVIDIA RTX 3080), adaptation took 5–24 min at baseline and 15–65 min in the final setting, about three times as long. Reading one segment took two to four times as long, but at most 25 ms.
Reading 100,000 segments in batches took 14–26 times as long, 4–24 min.

On another computer the times differ. The ratios for adaptation and for many segments follow from the amount of computation and should be similar;
the ratio for one segment depends also on the processor.

In [ ]:
def _columns(table, value, name, unit):
    """Baseline, final setting and ratio of one kind of time, as columns of the summary (one row for each foundation model)."""
    wide = table.pivot(index="model", columns="setting", values=value).reindex(index=MODELS, columns=["baseline", "final"])
    ratio = table[table.setting == "final"].set_index("model").ratio_to_baseline.reindex(MODELS)
    return pd.DataFrame({(name, f"baseline ({unit})"): wide["baseline"], (name, f"final ({unit})"): wide["final"], (name, "ratio"): ratio})


_adaptation = "Adaptation" + (" (QUICK TEST: not a measurement)" if QUICK_TEST else "")
SUMMARY_TABLE = pd.concat([_columns(ADAPTATION[ADAPTATION.part == "total"], "minutes", _adaptation, "min"),
                           _columns(READING[READING.task == "one_segment"], "milliseconds_per_segment", "One segment", "ms"),
                           _columns(READING[READING.task == "many_segments"], "minutes_per_100000_segments", f"{N_SEGMENTS:,} segments", "min")], axis=1)
SUMMARY_TABLE.index = [MODEL_NAMES[m] for m in SUMMARY_TABLE.index]
for _name in SUMMARY_TABLE.columns.get_level_values(0).unique():
    _t = SUMMARY_TABLE[_name]; _unit = _t.columns[0].split("(")[1].rstrip(")")
    print(f"{_name}: baseline {_t.iloc[:, 0].min():.1f}–{_t.iloc[:, 0].max():.1f} {_unit}, final setting {_t.iloc[:, 1].min():.1f}–{_t.iloc[:, 1].max():.1f} {_unit}, "
          f"{_t.iloc[:, 2].min():.1f}–{_t.iloc[:, 2].max():.1f} times as long")
SUMMARY_TABLE.round(1)

## Files written

Both files are in `TIMING`. Notebook 08 draws Supplementary Fig. 3 from them.

### `adaptation_time.csv`

One row = one part of the adaptation of one foundation model in one setting, or the total of a setting (`part` = `total`).
When everything is measured the file has 21 rows: for each of the three foundation models, 2 rows of the baseline (`fine_tuning`, `total`) and 5 rows of the final setting.

| Column | Content |
|---|---|
| `model` | foundation model: `ecgfounder`, `hubert` or `ecgfm` (the names in `MODELS`) |
| `setting` | `baseline` or `final` |
| `part` | baseline: `fine_tuning` (the one fine-tuning run). Final setting: `teacher_run`, `teacher_outputs` (calculation of the teacher outputs), `student_run`, `head_fitting` (fitting of the penalized head with the penalty weight `PENALTY_WEIGHT`). Both settings: `total`, the sum of the parts of the setting |
| `minutes` | time of the part in minutes, measured once |
| `ratio_to_baseline` | in the rows `total` only: `minutes` divided by the `minutes` of the total of the baseline of the same model (1 for the baseline); empty in the other rows |
| `peak_gpu_gb` | largest amount of memory that PyTorch had allocated on the graphics card during the part, in GB (2³⁰ bytes); in the rows `total`, the maximum over the parts; empty where it was not measured |
| `gpu` | name and memory of the graphics card |
| `quick_test` | `False` in a measurement; `True` only in `adaptation_time_quick_test.csv` |

Supplementary Fig. 3a: the bar of the baseline is `minutes` of the row (`baseline`, `total`). The bar of the final setting is stacked from `teacher_run` + `teacher_outputs` (drawn together as the teacher run), `student_run` and `head_fitting`;
the numbers next to it are `minutes` and `ratio_to_baseline` of the row (`final`, `total`). The peak memory in adaptation is the largest `peak_gpu_gb` of the rows with `setting` = `final`.

### `adaptation_time_quick_test.csv`

Written instead of `adaptation_time.csv` when the notebook runs with `QUICK_TEST = True`. Same columns, with `quick_test` = `True`.
Its times are those of 40 steps on a few hundred recordings: they show that the code runs and must not be used in a figure.

### `reading_time.csv`

One row = one task of reading with one foundation model in one setting: 3 models × 2 tasks × 2 settings = 12 rows.
The file is written under this name in both modes of `QUICK_TEST`, because the reading times are measured in full in both.

| Column | Content |
|---|---|
| `model` | foundation model, as above |
| `task` | `one_segment`: segments are read one at a time. `many_segments`: segments are read in batches of 512 |
| `setting` | `baseline` (one read of the segment, model head) or `final` (mixed reading, penalized head) |
| `forward_passes` | forward passes of the network for each segment: 1 in the baseline, 26 in the final setting (the segment and its 25 windows) |
| `segments_per_batch` | segments that are read together: 1 or 512 |
| `milliseconds_per_segment` | time to read one segment, in milliseconds. `one_segment`: median over 7 rounds of the median of 100 reads. `many_segments`: median of three times to read 1,024 segments, divided by 1,024 |
| `ratio_to_baseline` | time relative to the baseline of the same model and task (1 for the baseline). `one_segment`: median over the rounds of the ratio within a round, so it can differ slightly from the ratio of the two values of `milliseconds_per_segment`. `many_segments`: ratio of the two values of `milliseconds_per_segment` |
| `minutes_per_100000_segments` | rows `many_segments` only: time to read 100,000 segments with a batch size of 512, in minutes = `milliseconds_per_segment` × 100,000 / 60,000; empty in the rows `one_segment` |
| `peak_gpu_gb` | largest amount of memory that PyTorch had allocated on the graphics card during the reading, in GB (weights, inputs and activations; memory that was allocated before the model was loaded is not counted) |
| `gpu` | name and memory of the graphics card |

Supplementary Fig. 3b: rows with `task` = `one_segment`; the bars are `milliseconds_per_segment` of the baseline and of the final setting, and the ratio is `ratio_to_baseline` of the final setting.
Supplementary Fig. 3c: rows with `task` = `many_segments`; the bars are `minutes_per_100000_segments`, and the ratio is `ratio_to_baseline` of the final setting.
The peak memory in reading is the largest `peak_gpu_gb` of the rows with `setting` = `final`.

### Folder `adaptation_runs`

The models (`models/{tag}.pt`), training logs, teacher outputs and penalized heads of the timed runs (with `QUICK_TEST = True`: `adaptation_runs_quick_test`). About 2.4 GB in either mode.
No other notebook reads this folder, and it can be deleted when this notebook has finished.

In [ ]:
for _f in (ADAPTATION_FILE, READING_FILE):
    print(f"{_f.name}: {len(pd.read_csv(_f))} rows, {_f.stat().st_size / 1024:.1f} KB")
_files = [f for f in RUN_DIR.rglob("*") if f.is_file()]
print(f"{RUN_DIR.name}: {len(_files)} files, {sum(f.stat().st_size for f in _files) / 2 ** 30:.2f} GB")